# PWV04 : AuxTel / Spectractor vs MERRA-2 (TQV) — Cross-validation

## Overview

This notebook compares the precipitable water vapour (PWV) measured by
AuxTel/Spectractor with the total precipitable water column (TQV, in kg m⁻²
= mm) from the **MERRA-2** reanalysis product `inst1_2d_asm_Nx`, available at
hourly cadence at the grid point closest to Cerro Pachon.

### Goals

1. **Temporal matching** — for each AuxTel observation, find the closest
   MERRA-2 timestamp and record the signed time difference
   $\Delta t_{\mathrm{match}}$ (minutes).  This quantifies how well the two
   datasets are co-temporal.

2. **Global correlation** — scatter plot and Pearson/Spearman correlation
   coefficients, correlation ellipse, and linear regression between
   $\mathrm{PWV_{AuxTel}}$ and $\mathrm{TQV_{MERRA-2}}$.

3. **Seasonal dependence** — split the matched dataset into the two
   Southern-Hemisphere seasons expected to differ most in PWV:
   - **Dry season (Winter JJA)** : June–August — austral winter, low PWV.
   - **Wet season (Summer DJF)** : December–February — austral summer, high PWV.
   Compare correlation ellipses and regression slopes for each season.

4. **Collimator cut** — a `FLAG_WITHCOLLIMATOR` flag (set at the top) allows
   the user to restrict the analysis to observations taken after the AuxTel
   collimator was installed (2023-09-30), which provides more repeatable
   PWV measurements.

### Physical background

MERRA-2 TQV is the vertically integrated water vapour in the entire
atmospheric column, expressed in kg m⁻² (numerically equal to mm of
precipitable water under standard conditions).  Because MERRA-2 is a
reanalysis product with ~50 km horizontal resolution and 1-hour temporal
resolution, it cannot resolve the rapid intra-night fluctuations seen by
AuxTel.  The expected correlation is therefore strong on synoptic and seasonal
scales but degrades at short timescales.  A slope significantly different from
unity would indicate a systematic bias between the two datasets (e.g. a column
correction factor, or the effect of the MERRA-2 grid not being co-located with
the telescope).

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-19  
**Kernel :** conda_py313

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator
import scipy.stats as stats
from scipy.optimize import curve_fit
from astropy.time import Time
from collections import OrderedDict
import seaborn as sns

%matplotlib inline

# ── publication rc params ─────────────────────────────────────────────────────
mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict, legendtag,
    butlerusercollectiondict,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    filename_m2,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
# ── output paths ──────────────────────────────────────────────────────────────
pathfigs = "figs_PWV04merra"
pathdata = "data_PWV01seas"      # reuse cuts from PWV01
prefix   = "pwv04merra"
os.makedirs(pathfigs, exist_ok=True)
figtype  = ".pdf"

---
## 1. User flags

Set these before running the notebook.

In [ ]:
# ── Collimator cut ────────────────────────────────────────────────────────────
# Set True to restrict to observations AFTER the collimator was installed.
# The collimator removes sky background and gives more repeatable PWV values.
# The user can toggle this without changing anything else in the notebook.
FLAG_WITHCOLLIMATOR    = True
datetime_WITHCOLLIMATOR = pd.to_datetime("2023-09-30 00:00:00+0000", utc=True)

# ── Quality cut level ────────────────────────────────────────────────────────
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

# ── Maximum allowed |Delta_t| between AuxTel obs and nearest MERRA-2 point ──
# MERRA-2 cadence is 1 h = 60 min, so any match is within 30 min at most.
# Keeping only |Delta_t| < MAX_DT_MATCH_MIN rejects isolated observations
# far from any MERRA-2 timestamp (should never happen with hourly data).
MAX_DT_MATCH_MIN = 35.0   # minutes

print(f"Collimator cut : {FLAG_WITHCOLLIMATOR}  (after {datetime_WITHCOLLIMATOR.date()})")
print(f"Quality cuts   : {'loose' if FLAG_LOOSE_CUTS else 'tight'}")
print(f"Max match lag  : {MAX_DT_MATCH_MIN} min")

In [ ]:
#Cut bad PWV from Merra2
PWVMIN = 0.
PWVMAX = 16.

---
## 2. Load AuxTel / Spectractor data

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading AuxTel data: {atmfilename}")

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

# ── Time from DATE-OBS ────────────────────────────────────────────────────────
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)
df_spec = df_spec.sort_values("Time", ascending=True).reset_index(drop=True)

# ── Rename Corentin columns ───────────────────────────────────────────────────
if "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

# ── Re-compute nightObs from local Chile time (id column is buggy) ────────────
df_spec["Time_local"] = df_spec["Time"].dt.tz_convert("America/Santiago")
df_spec["nightObs"]   = (
    (df_spec["Time_local"] - pd.Timedelta(hours=12))
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# ── Filter selection ──────────────────────────────────────────────────────────
if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST)].copy()

pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum", "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f"Shape after filter selection: {df_spec.shape}")
print(f"Date range: {df_spec['Time'].min().date()}  ->  {df_spec['Time'].max().date()}")
print("Filters:", df_spec["FILTER"].unique())

In [ ]:
# ── chi2 normalisation (needed for quality cuts) ──────────────────────────────
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="CHI2_FIT", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_ram", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_rum", ext="norm")

In [ ]:
if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec["Time"] > datetime_WITHCOLLIMATOR ]

In [ ]:
# ── Quality cuts ──────────────────────────────────────────────────────────────
if FLAG_LOOSE_CUTS:
    filename_cuts = f"{pathdata}/cuts_loose_finaldecision.json"
    cutstype_name = "loose-cuts"
elif FLAG_TIGHT_CUTS:
    filename_cuts = f"{pathdata}/cuts_tight_finaldecision.json"
    cutstype_name = "tight-cuts"
else:
    filename_cuts = f"{pathdata}/cuts_finaldecision.json"
    cutstype_name = "standard-cuts"

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col="id")
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on="id")
df_keep     = df_selected[df_selected["pass_all_cuts"]].copy()

print(f"Quality cuts ({cutstype_name}): {len(df_keep)} / {len(df_spec)} kept")

In [ ]:
# ── Collimator cut (applied by user flag) ─────────────────────────────────────
if FLAG_WITHCOLLIMATOR:
    n_before = len(df_keep)
    df_keep  = df_keep[df_keep["Time"] >= datetime_WITHCOLLIMATOR].copy()
    print(f"Collimator cut: {len(df_keep)} / {n_before} kept "
          f"(after {datetime_WITHCOLLIMATOR.date()})")
else:
    print("No collimator cut applied.")

# Best-estimate PWV from AuxTel
#df_keep["PWV_auxtel"]     = 0.5 * (df_keep["PWV [mm]_ram"] + df_keep["PWV [mm]_rum"])
#df_keep["PWV_auxtel_err"] = np.sqrt(
#    df_keep["PWV [mm]_err_ram"]**2 + df_keep["PWV [mm]_err_rum"]**2
#)

# Best-estimate PWV from AuxTel
df_keep["PWV_auxtel"]     = df_keep["PWV [mm]_ram"] 
df_keep["PWV_auxtel_err"] = df_keep["PWV [mm]_err_ram"]


# Calendar fields
df_keep["year"]  = df_keep["Time"].dt.year
df_keep["month"] = df_keep["Time"].dt.month

print(f"Final AuxTel sample: {len(df_keep)} observations")
print(f"Date range: {df_keep['Time'].min().date()}  ->  {df_keep['Time'].max().date()}")

---
## 3. Load MERRA-2 data

MERRA-2 `inst1_2d_asm_Nx` product — hourly instantaneous fields at the
Cerro Pachon grid point.  `TQV` (Total precipitable Water Vapour) is in
kg m⁻², numerically equal to mm of PWV.

In [ ]:
print(f"Loading MERRA-2 data: {filename_m2}")
df_merra = pd.read_csv(filename_m2)
df_merra["Time"] = pd.to_datetime(df_merra["time"], utc=True)
df_merra = df_merra.sort_values("Time").reset_index(drop=True)

print(f"MERRA-2 shape: {df_merra.shape}")
print(f"MERRA-2 date range: {df_merra['Time'].min().date()}  ->  {df_merra['Time'].max().date()}")
print(f"TQV range: {df_merra['TQV'].min():.2f} – {df_merra['TQV'].max():.2f} mm")
print("\nColumns:", list(df_merra.columns))

In [ ]:
# Quick overview of MERRA-2 TQV
fig0, ax0 = plt.subplots(figsize=(16, 4))
ax0.plot(df_merra["Time"], df_merra["TQV"],
         lw=0.5, color="steelblue", alpha=0.8)
ax0.set_xlabel("Date (UTC)", fontsize=12)
ax0.set_ylabel("TQV [mm]", fontsize=12)
ax0.set_title("MERRA-2 TQV at Cerro Pachon — full time series", fontsize=11)
fig0.tight_layout()
plt.show()

---
## 4. Temporal matching — nearest MERRA-2 point for each AuxTel observation

### Method

MERRA-2 has a fixed 1-hour cadence, so any AuxTel observation is at most
30 minutes away from a MERRA-2 timestamp.  We use
`pandas.merge_asof` with `direction='nearest'`, which performs an
efficient sorted merge and assigns to each AuxTel row the MERRA-2 row
whose timestamp is closest in time.  The signed time offset
$$
\Delta t_{\mathrm{match}} = t_{\mathrm{AuxTel}} - t_{\mathrm{MERRA-2}}
$$
is recorded in minutes.  A positive value means the AuxTel observation
was taken *after* the nearest MERRA-2 timestamp, negative *before*.

Rows with $|\Delta t_{\mathrm{match}}| > 35$ min are flagged and excluded
(should not occur with hourly MERRA-2 data, but provides a safety guard).

In [ ]:
# ── Prepare the two sorted dataframes for merge_asof ─────────────────────────
df_aux_sorted   = df_keep.sort_values("Time").reset_index(drop=True)
df_merra_sorted = df_merra[["Time", "TQV"]].copy().sort_values("Time").reset_index(drop=True)

# ── Nearest-neighbour match ───────────────────────────────────────────────────
# merge_asof with direction='nearest' finds the closest MERRA-2 row for
# each AuxTel row (both dataframes must be sorted by the key column).
df_matched = pd.merge_asof(
    df_aux_sorted,
    df_merra_sorted.rename(columns={"Time": "Time_m2", "TQV": "TQV_m2"}),
    left_on  = "Time",
    right_on = "Time_m2",
    direction= "nearest",
    suffixes = ("", "_m2")
)

# ── Signed time difference in minutes ────────────────────────────────────────
df_matched["dt_match_min"] = (
    (df_matched["Time"] - df_matched["Time_m2"])
    .dt.total_seconds() / 60.0
)

# ── Apply maximum lag filter ──────────────────────────────────────────────────
n_before = len(df_matched)
df_matched = df_matched[df_matched["dt_match_min"].abs() <= MAX_DT_MATCH_MIN].copy()

print(f"Matched pairs         : {len(df_matched)} / {n_before}")
print(f"dt_match range        : {df_matched['dt_match_min'].min():.1f} to "
      f"{df_matched['dt_match_min'].max():.1f} min")
print(f"dt_match median       : {df_matched['dt_match_min'].median():.1f} min")
print(f"TQV_m2 range          : {df_matched['TQV_m2'].min():.2f} – "
      f"{df_matched['TQV_m2'].max():.2f} mm")
print(f"PWV_auxtel range      : {df_matched['PWV_auxtel'].min():.2f} – "
      f"{df_matched['PWV_auxtel'].max():.2f} mm")

In [ ]:
# ── Distribution of dt_match ──────────────────────────────────────────────────
fig1, axes1 = plt.subplots(1, 2, figsize=(14, 4))

axes1[0].hist(df_matched["dt_match_min"], bins=60,
              color="steelblue", alpha=0.85, edgecolor="white", linewidth=0.3)
axes1[0].axvline(0, color="black", ls="--", lw=1)
axes1[0].set_xlabel(r"$\Delta t_{\mathrm{match}}$ [min]", fontsize=12)
axes1[0].set_ylabel("Count", fontsize=12)
axes1[0].set_title("Distribution of AuxTel – MERRA-2 time offset", fontsize=11)

axes1[1].hist(df_matched["dt_match_min"].abs(), bins=35,
              color="darkorange", alpha=0.85, edgecolor="white", linewidth=0.3)
axes1[1].set_xlabel(r"$|\Delta t_{\mathrm{match}}|$ [min]", fontsize=12)
axes1[1].set_ylabel("Count", fontsize=12)
axes1[1].set_title("Absolute time offset to nearest MERRA-2 point", fontsize=11)

fig1.suptitle(f"Temporal matching quality — {version_run}", fontsize=12, y=1.02)
fig1.tight_layout()
figfile1 = f"{pathfigs}/{prefix}_{version_run}_dt_match_hist{figtype}"
fig1.savefig(figfile1, bbox_inches="tight")
print(f"Saved: {figfile1}")
plt.show()

---
## 5. Seasonal labelling

We use the Southern-Hemisphere season convention (same as PWV02):

| Season | Months | Expected PWV |
|--------|--------|--------------|
| Summer (DJF) | Dec, Jan, Feb | High |
| Autumn (MAM) | Mar, Apr, May | Medium |
| Winter (JJA) | Jun, Jul, Aug | Low |
| Spring (SON) | Sep, Oct, Nov | Medium |

For the correlation study we focus on **Winter (dry)** vs **Summer (wet)**
as the two extremes.

In [ ]:
SEASON_MAP = {
    12: "Summer (DJF)",  1: "Summer (DJF)",  2: "Summer (DJF)",
     3: "Autumn (MAM)",  4: "Autumn (MAM)",  5: "Autumn (MAM)",
     6: "Winter (JJA)",  7: "Winter (JJA)",  8: "Winter (JJA)",
     9: "Spring (SON)", 10: "Spring (SON)", 11: "Spring (SON)",
}
SEASON_ORDER  = ["Summer (DJF)", "Autumn (MAM)", "Winter (JJA)", "Spring (SON)"]
SEASON_COLORS = {
    "Summer (DJF)": "#d62728",   # red
    "Autumn (MAM)": "#ff7f0e",   # orange
    "Winter (JJA)": "#1f77b4",   # blue
    "Spring (SON)": "#2ca02c",   # green
}

df_matched["season"] = df_matched["month"].map(SEASON_MAP)

print("Observations per season:")
print(df_matched["season"].value_counts()[SEASON_ORDER])

---
## 6. Helper functions — correlation ellipse & linear regression

In [ ]:
def correlation_ellipse(ax, x, y, n_std=2.0, color="steelblue",
                        alpha_fill=0.15, alpha_edge=0.8, lw=1.5, label=None):
    """
    Draw the n_std-sigma confidence ellipse of the bivariate distribution (x, y).

    The ellipse is derived from the 2x2 covariance matrix.  Its axes are
    aligned with the eigenvectors of the covariance matrix and scaled by
    n_std standard deviations.

    Parameters
    ----------
    n_std : float
        Number of standard deviations (1 -> ~39% confidence, 2 -> ~86%).

    Returns
    -------
    ellipse : matplotlib.patches.Ellipse
    """
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 3:
        return None

    cov  = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)

    ell = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=width, height=height, angle=theta,
        facecolor=color, edgecolor=color,
        alpha=alpha_fill, linewidth=0, zorder=4
    )
    ax.add_patch(ell)

    ell_edge = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=width, height=height, angle=theta,
        facecolor="none", edgecolor=color,
        alpha=alpha_edge, linewidth=lw, zorder=5,
        label=label
    )
    ax.add_patch(ell_edge)
    return ell_edge


def linear_fit(x, y, yerr=None):
    """
    Ordinary least-squares linear regression y = a + b*x.

    Returns
    -------
    slope, intercept, slope_err, intercept_err, r_value (Pearson), p_value
    """
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    res  = stats.linregress(x, y)
    return res.slope, res.intercept, res.stderr, res.intercept_stderr, res.rvalue, res.pvalue


def print_corr_stats(label, x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)
    slope, intercept, slope_err, int_err, r_lin, pval = linear_fit(x, y)
    print(f"  [{label}]  N={len(x)}")
    print(f"    Pearson r  = {r_p:.4f}  (p={p_p:.2e})")
    print(f"    Spearman r = {r_s:.4f}  (p={p_s:.2e})")
    print(f"    OLS:  slope = {slope:.4f} +/- {slope_err:.4f}")
    print(f"          intercept = {intercept:.4f} +/- {int_err:.4f} mm")
    return dict(label=label, n=len(x),
                pearson_r=r_p, pearson_p=p_p,
                spearman_r=r_s, spearman_p=p_s,
                slope=slope, slope_err=slope_err,
                intercept=intercept, intercept_err=int_err)

---
## 7. Global correlation: AuxTel PWV vs MERRA-2 TQV

In [ ]:
x_all = df_matched["TQV_m2"].values
y_all = df_matched["PWV_auxtel"].values
e_all = df_matched["PWV_auxtel_err"].values

print("=== Global correlation ===")
stats_all = print_corr_stats("All data", x_all, y_all)

slope_all = stats_all["slope"]
icpt_all  = stats_all["intercept"]

In [ ]:
# ── Scatter + ellipse + regression ────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 7))

ax2.errorbar(
    x_all, y_all, yerr=e_all,
    fmt="o", color="steelblue", ms=2.5, alpha=0.4,
    elinewidth=0.4, capsize=0, zorder=3, label="All observations"
)

# 1-sigma and 2-sigma ellipses
correlation_ellipse(ax2, x_all, y_all, n_std=1.0,
                    color="steelblue", alpha_fill=0.20, label=r"$1\sigma$ ellipse")
correlation_ellipse(ax2, x_all, y_all, n_std=2.0,
                    color="steelblue", alpha_fill=0.08, alpha_edge=0.5,
                    label=r"$2\sigma$ ellipse")

# OLS regression line
xrange = np.linspace(x_all.min(), x_all.max(), 200)
ax2.plot(xrange, slope_all * xrange + icpt_all,
         color="darkred", lw=2, zorder=6,
         label=f"OLS: slope={slope_all:.3f}, offset={icpt_all:.2f} mm")

# 1:1 reference line
xy_max = max(x_all.max(), y_all.max()) * 1.05
ax2.plot([0, xy_max], [0, xy_max],
         color="gray", lw=1.0, ls="--", zorder=2, label="1:1 line")

ax2.set_xlabel("MERRA-2 TQV [mm]", fontsize=13)
ax2.set_ylabel("AuxTel PWV [mm]", fontsize=13)
ax2.set_title(
    f"AuxTel vs MERRA-2 PWV — all seasons\n"
    f"Pearson r = {stats_all['pearson_r']:.3f}  "
    f"(N = {stats_all['n']})",
    fontsize=11
)
ax2.set_xlim(PWVMIN, PWVMAX)
ax2.set_ylim(PWVMIN, PWVMAX)
ax2.legend(fontsize=9, loc="upper left")

figfile2 = f"{pathfigs}/{prefix}_{version_run}_scatter_global{figtype}"
fig2.savefig(figfile2, bbox_inches="tight")
print(f"Saved: {figfile2}")
plt.show()

---
## 8. Seasonal correlation — all four seasons

In [ ]:
print("=== Correlation statistics per season ===")
season_stats = {}
for season in SEASON_ORDER:
    sub = df_matched[df_matched["season"] == season]
    if len(sub) < 5:
        print(f"  {season}: too few points, skipped")
        continue
    s = print_corr_stats(season,
                         sub["TQV_m2"].values,
                         sub["PWV_auxtel"].values)
    season_stats[season] = s
    print()

In [ ]:
# ── 2x2 panel: one scatter per season ─────────────────────────────────────────
fig3, axes3 = plt.subplots(2, 2, figsize=(14, 13))
axes3_flat  = axes3.flatten()

for i, season in enumerate(SEASON_ORDER):
    ax = axes3_flat[i]
    sub = df_matched[df_matched["season"] == season]
    col = SEASON_COLORS[season]

    if len(sub) < 5:
        ax.text(0.5, 0.5, "Not enough data",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title(season)
        continue

    x_s = sub["TQV_m2"].values
    y_s = sub["PWV_auxtel"].values
    e_s = sub["PWV_auxtel_err"].values

    ax.errorbar(x_s, y_s, yerr=e_s,
                fmt="o", color=col, ms=2.5, alpha=0.45,
                elinewidth=0.4, capsize=0, zorder=3)

    correlation_ellipse(ax, x_s, y_s, n_std=1.0, color=col,
                        alpha_fill=0.20, label=r"$1\sigma$")
    correlation_ellipse(ax, x_s, y_s, n_std=2.0, color=col,
                        alpha_fill=0.08, alpha_edge=0.5, label=r"$2\sigma$")

    sl = season_stats.get(season, {}).get("slope", np.nan)
    ic = season_stats.get(season, {}).get("intercept", np.nan)
    rr = season_stats.get(season, {}).get("pearson_r", np.nan)

    if np.isfinite(sl):
        xr = np.linspace(max(x_s.min(), 0), x_s.max(), 200)
        ax.plot(xr, sl * xr + ic, color="darkred", lw=1.8, zorder=6,
                label=f"slope={sl:.3f}")

    xy_max_s = max(x_s.max(), y_s.max()) * 1.05
    ax.plot([0, xy_max_s], [0, xy_max_s],
            color="gray", lw=1.0, ls="--", zorder=2)

    ax.set_xlabel("MERRA-2 TQV [mm]", fontsize=11)
    ax.set_ylabel("AuxTel PWV [mm]", fontsize=11)
    ax.set_title(
        f"{season}\n"
        f"Pearson r = {rr:.3f}  (N = {len(sub)})",
        fontsize=10
    )
    ax.set_xlim(PWVMIN, PWVMAX)
    ax.set_ylim(PWVMIN, PWVMAX)
    ax.legend(fontsize=8, loc="upper left")

fig3.suptitle(
    f"AuxTel vs MERRA-2 PWV — per season\n{version_run}",
    fontsize=12, y=1.01
)
fig3.tight_layout()
figfile3 = f"{pathfigs}/{prefix}_{version_run}_scatter_per_season{figtype}"
fig3.savefig(figfile3, bbox_inches="tight")
print(f"Saved: {figfile3}")
plt.show()

---
## 9. Dry vs Wet season comparison — overlaid ellipses

Superimposed scatter + ellipses for **Winter (JJA, dry)** and
**Summer (DJF, wet)** to highlight the seasonal dependence of the
AuxTel–MERRA-2 correlation.

In [ ]:
DRY_SEASON = "Winter (JJA)"
WET_SEASON = "Summer (DJF)"

df_dry = df_matched[df_matched["season"] == DRY_SEASON]
df_wet = df_matched[df_matched["season"] == WET_SEASON]

fig4, ax4 = plt.subplots(figsize=(9, 8))

for df_s, season, col in [
    (df_wet, WET_SEASON, SEASON_COLORS[WET_SEASON]),
    (df_dry, DRY_SEASON, SEASON_COLORS[DRY_SEASON]),
]:
    x_s = df_s["TQV_m2"].values
    y_s = df_s["PWV_auxtel"].values
    e_s = df_s["PWV_auxtel_err"].values

    ax4.errorbar(x_s, y_s, yerr=e_s,
                 fmt="o", color=col, ms=2.5, alpha=0.35,
                 elinewidth=0.35, capsize=0, zorder=3)

    correlation_ellipse(ax4, x_s, y_s, n_std=1.0, color=col,
                        alpha_fill=0.18, label=f"{season} ($1\\sigma$)")
    correlation_ellipse(ax4, x_s, y_s, n_std=2.0, color=col,
                        alpha_fill=0.06, alpha_edge=0.45,
                        label=f"{season} ($2\\sigma$)")

    sl = season_stats.get(season, {}).get("slope", np.nan)
    ic = season_stats.get(season, {}).get("intercept", np.nan)
    rr = season_stats.get(season, {}).get("pearson_r", np.nan)
    if np.isfinite(sl):
        xr = np.linspace(max(x_s.min(), 0), x_s.max(), 200)
        ax4.plot(xr, sl * xr + ic,
                 color=col, lw=2.0, ls="-", zorder=6,
                 label=f"{season} OLS (r={rr:.3f}, slope={sl:.3f})")

# 1:1 line
xy_max4 = max(
    df_matched["TQV_m2"].max(),
    df_matched["PWV_auxtel"].max()
) * 1.05
ax4.plot([0, xy_max4], [0, xy_max4],
         color="gray", lw=1.0, ls="--", zorder=2, label="1:1 line")

ax4.set_xlabel("MERRA-2 TQV [mm]", fontsize=13)
ax4.set_ylabel("AuxTel PWV [mm]", fontsize=13)
ax4.set_title(
    f"Dry (JJA) vs Wet (DJF) seasons\n"
    f"N(dry)={len(df_dry)}, N(wet)={len(df_wet)}",
    fontsize=11
)
ax4.set_xlim(PWVMIN, PWVMAX)
ax4.set_ylim(PWVMIN, PWVMAX)
ax4.legend(fontsize=8.5, loc="upper left")

figfile4 = f"{pathfigs}/{prefix}_{version_run}_scatter_dry_vs_wet{figtype}"
fig4.savefig(figfile4, bbox_inches="tight")
print(f"Saved: {figfile4}")
plt.show()

---
## 10. Time series: AuxTel and MERRA-2 vs MJD

Overlay both time series to visually assess agreement and identify
systematic offsets.

In [ ]:
def make_date_axis(ax_mjd, mjd_min, mjd_max, n_ticks=8):
    ax_date = ax_mjd.twiny()
    ax_date.set_xlim(ax_mjd.get_xlim())
    tick_mjds   = np.linspace(mjd_min, mjd_max, n_ticks)
    tick_labels = [
        Time(m, format='mjd', scale='utc').strftime('%Y-%m')
        for m in tick_mjds
    ]
    ax_date.set_xticks(tick_mjds)
    ax_date.set_xticklabels(tick_labels, rotation=30, ha='left', fontsize=9)
    ax_date.set_xlabel('Date (UTC)', fontsize=10, labelpad=6)
    return ax_date


mjd_min = df_matched["MJD"].min()
mjd_max = df_matched["MJD"].max()

fig5, axes5 = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# -- top: both time series overlaid
axes5[0].errorbar(
    df_matched["MJD"], df_matched["PWV_auxtel"],
    yerr=df_matched["PWV_auxtel_err"],
    fmt="o", color="steelblue", ms=2, alpha=0.5,
    elinewidth=0.4, capsize=0, zorder=4, label="AuxTel PWV"
)
axes5[0].scatter(
    df_matched["MJD"], df_matched["TQV_m2"],
    marker="x", color="darkorange", s=15, alpha=0.6,
    zorder=3, label="MERRA-2 TQV"
)
axes5[0].set_ylabel("PWV / TQV [mm]", fontsize=12)
axes5[0].set_title("AuxTel PWV and MERRA-2 TQV vs time", fontsize=11)
axes5[0].legend(fontsize=10)
axes5[0].set_ylim(PWVMIN, PWVMAX)
make_date_axis(axes5[0], mjd_min, mjd_max, n_ticks=10)

# -- bottom: residual AuxTel - MERRA-2 coloured by season
df_matched["residual"] = df_matched["PWV_auxtel"] - df_matched["TQV_m2"]

for season in SEASON_ORDER:
    sub = df_matched[df_matched["season"] == season]
    axes5[1].scatter(
        sub["MJD"], sub["residual"],
        color=SEASON_COLORS[season], s=8, alpha=0.55,
        label=season, zorder=3
    )

axes5[1].axhline(0, color="black", lw=1.0, ls="--")
axes5[1].axhline(
    df_matched["residual"].median(), color="gray", lw=1.2, ls=":",
    label=f"Median = {df_matched['residual'].median():.2f} mm"
)
axes5[1].set_xlabel("MJD", fontsize=12)
axes5[1].set_ylabel("AuxTel – MERRA-2 [mm]", fontsize=12)
axes5[1].set_title("Residual (AuxTel – MERRA-2) coloured by season", fontsize=11)
axes5[1].legend(fontsize=9, ncol=2, loc="upper right")

fig5.suptitle(
    f"AuxTel vs MERRA-2 — time series and residuals\n{version_run}",
    fontsize=12, y=1.01
)
fig5.tight_layout()
figfile5 = f"{pathfigs}/{prefix}_{version_run}_timeseries{figtype}"
fig5.savefig(figfile5, bbox_inches="tight")
print(f"Saved: {figfile5}")
plt.show()

---
## 11. Residual histograms per season

In [ ]:
fig6, ax6 = plt.subplots(figsize=(8, 5))

for season in SEASON_ORDER:
    sub = df_matched[df_matched["season"] == season]
    if len(sub) < 5:
        continue
    ax6.hist(
        sub["residual"], bins=40,
        color=SEASON_COLORS[season], alpha=0.55,
        label=f"{season}  (med={sub['residual'].median():.2f} mm,"
              f" std={sub['residual'].std():.2f} mm)",
        edgecolor="white", linewidth=0.3
    )

ax6.axvline(0, color="black", lw=1.2, ls="--")
ax6.set_xlabel("AuxTel – MERRA-2 TQV [mm]", fontsize=12)
ax6.set_ylabel("Count", fontsize=12)
ax6.set_title("Residual distributions per season", fontsize=11)
ax6.legend(fontsize=9)

figfile6 = f"{pathfigs}/{prefix}_{version_run}_residual_hist{figtype}"
fig6.savefig(figfile6, bbox_inches="tight")
print(f"Saved: {figfile6}")
plt.show()

---
## 12. Summary table — statistics per season

In [ ]:
rows = []
for season in SEASON_ORDER:
    sub = df_matched[df_matched["season"] == season]
    if len(sub) < 5:
        continue
    r_p, _ = stats.pearsonr(sub["TQV_m2"], sub["PWV_auxtel"])
    r_s, _ = stats.spearmanr(sub["TQV_m2"], sub["PWV_auxtel"])
    sl, ic, sl_e, ic_e, _, _ = linear_fit(
        sub["TQV_m2"].values, sub["PWV_auxtel"].values
    )
    res = sub["residual"]
    rows.append({
        "Season"       : season,
        "N"            : len(sub),
        "Pearson r"    : round(r_p, 4),
        "Spearman r"   : round(r_s, 4),
        "OLS slope"    : round(sl, 4),
        "slope err"    : round(sl_e, 4),
        "OLS offset [mm]" : round(ic, 3),
        "Resid. med [mm]" : round(res.median(), 3),
        "Resid. std [mm]" : round(res.std(), 3),
    })

df_stats_season = pd.DataFrame(rows)
display(df_stats_season.style
    .format({c: "{:.4f}" for c in ["Pearson r", "Spearman r",
                                    "OLS slope", "slope err"]})
    .format({"N": "{:d}"})
    .set_caption(f"Correlation statistics per season — {version_run}")
    .set_table_styles([
        {"selector": "th",
         "props": [("background-color", "#dce6f1"), ("font-weight", "bold")]},
        {"selector": "tr:nth-child(even)",
         "props": [("background-color", "#f5f8fc")]},
    ])
)

---
## 13. Export results

In [ ]:
# Save matched dataframe (AuxTel + nearest MERRA-2 TQV + metadata)
cols_save = [
    "id", "nightObs", "FILTER", "TARGET",
    "Time", "MJD",
    "PWV [mm]_ram", "PWV [mm]_err_ram",
    "PWV_auxtel", "PWV_auxtel_err",
    "Time_m2", "TQV_m2", "dt_match_min",
    "season", "residual",
]
cols_save = [c for c in cols_save if c in df_matched.columns]

csv_out = f"{pathfigs}/{prefix}_{version_run}_matched_auxtel_merra2.csv"
df_matched[cols_save].to_csv(csv_out, index=False)
print(f"Saved matched dataset: {csv_out}")

# Save statistics JSON
results = {
    "version_run"        : version_run,
    "cuts_type"          : cutstype_name,
    "collimator_cut"     : FLAG_WITHCOLLIMATOR,
    "collimator_date"    : str(datetime_WITHCOLLIMATOR.date()),
    "max_dt_match_min"   : MAX_DT_MATCH_MIN,
    "N_matched"          : int(len(df_matched)),
    "global"             : stats_all,
    "per_season"         : {
        s: {
            k: float(v) if isinstance(v, (float, np.floating)) else v
            for k, v in d.items()
        }
        for s, d in season_stats.items()
    },
}

json_out = f"{pathfigs}/{prefix}_{version_run}_correlation_stats.json"
with open(json_out, "w") as fh:
    json.dump(results, fh, indent=2)
print(f"Saved statistics: {json_out}")
print(json.dumps(results, indent=2))